In [1]:
# %% [code] {"execution":{"iopub.status.busy":"2026-06-13T07:20:52.581489Z","iopub.execute_input":"2026-06-13T07:20:52.582389Z","iopub.status.idle":"2026-06-13T07:21:03.601230Z","shell.execute_reply.started":"2026-06-13T07:20:52.582346Z","shell.execute_reply":"2026-06-13T07:21:03.600123Z"}}
# ==============================================================================
# ARCHIVO: preprocess_andean_dataset.py
# FUNCIÓN: Preprocesamiento termodinámico, control de nulos y estrategia de anomalías
# ==============================================================================

import os
import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")

# Configuración gráfica de nivel científico para publicaciones (300 DPI)
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.dpi": 300,
    "font.family": "serif",
    "axes.labelsize": 10,
    "axes.titlesize": 11,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "figure.titlesize": 12
})

# ============================================================================
# PARÁMETROS GLOBALES DE CONFIGURACIÓN
# ============================================================================
INPUT_PATH = "/kaggle/input/datasets/danielchura/ema-dataset/IGP_EstacionEMA_data_2018_2025.csv"
OUTPUT_PATH = "dataset_huayao_preprocessed.csv"
OUTPUT_METRIC_STD = "metadata_desviacion_estandar.csv"
OUTPUT_METRIC_NULLS = "metadata_auditoria_nulos.csv"

# Fecha límite del set de entrenamiento para evitar fuga de información climatológica
TRAIN_END_DATE = "2023-12-31 23:00:00"


# ============================================================================
# INGENIERÍA DE CARACTERÍSTICAS Y PREPROCESAMIENTO CAUSAL
# ============================================================================

def load_and_sanitize_data(input_path: str) -> pd.DataFrame:
    """Carga el dataset crudo y valida que las variables físicas sean estrictamente numéricas."""
    if not os.path.exists(input_path):
        raise FileNotFoundError(f"No se encontró el archivo de entrada en {input_path}")

    df_raw = pd.read_csv(input_path)
    
    physical_cols = ["TT", "HR", "PP", "RR", "FF", "DD"]
    for col in physical_cols:
        if col in df_raw.columns:
            df_raw[col] = pd.to_numeric(df_raw[col], errors="coerce")
            
    return df_raw


def audit_variability_and_nulls(df_raw: pd.DataFrame) -> tuple[pd.Series, pd.DataFrame]:
    """Audita desviaciones estándar e inconsistencias antes de la unificación temporal."""
    candidates = ["year", "month", "day", "hour", "TT", "HR", "PP", "RR", "FF", "DD"]
    admin_cols = [col for col in ["UBIGEO", "FECHA_CORTE"] if col in df_raw.columns]
    all_eval_cols = candidates + admin_cols

    # Captura de nulos en origen
    original_nulls = df_raw[all_eval_cols].isnull().sum()
    variances = df_raw[all_eval_cols].std().fillna(0.0)
    constant_cols = [col for col in all_eval_cols if col in df_raw.columns and df_raw[col].nunique() <= 1]

    # Exportación de Tabla de Desviaciones
    df_std = pd.DataFrame({
        "Variable": variances.index, 
        "Desviacion_Estandar": variances.values.round(4)
    })
    df_std["Estado"] = np.where(df_std["Variable"].isin(admin_cols + constant_cols), "DESCARTE", "ACEPTADA")
    df_std.to_csv(OUTPUT_METRIC_STD, index=False)

    return original_nulls, df_std


def build_uniform_timeline(df_raw: pd.DataFrame, original_nulls: pd.Series) -> pd.DataFrame:
    """Construye una grilla temporal uniforme horaria y audita las brechas de datos resultantes."""
    df_raw["datetime"] = pd.to_datetime({
        "year": df_raw["year"],
        "month": df_raw["month"],
        "day": df_raw["day"],
        "hour": df_raw["hour"]
    })

    df = df_raw.drop_duplicates(subset=["datetime"]).set_index("datetime").sort_index()
    expected_range = pd.date_range(start=df.index.min(), end=df.index.max(), freq="1H")
    df = df.reindex(expected_range)

    candidates = ["year", "month", "day", "hour", "TT", "HR", "PP", "RR", "FF", "DD"]
    admin_cols = [col for col in ["UBIGEO", "FECHA_CORTE"] if col in df_raw.columns]
    all_eval_cols = candidates + admin_cols
    total_nulls = df[all_eval_cols].isnull().sum()

    # Tabla comparativa de valores faltantes por fallas de conexión (Gaps)
    df_nulls = pd.DataFrame({
        "Variable": all_eval_cols,
        "Nulos_Originales": original_nulls.values,
        "Nulos_por_Huecos": total_nulls.values - original_nulls.values,
        "Nulos_Totales": total_nulls.values,
        "Porcentaje_Total(%)": np.round((total_nulls.values / len(df)) * 100, 2)
    })
    df_nulls.to_csv(OUTPUT_METRIC_NULLS, index=False)

    return df


def apply_causal_imputation(df: pd.DataFrame) -> pd.DataFrame:
    """Aplica imputación causal hacia adelante respetando la dirección temporal."""
    if "RR" in df.columns:
        df["RR"] = df["RR"].fillna(0.0)

    cols_continuous = [col for col in ["TT", "HR", "PP", "FF", "DD"] if col in df.columns]
    df[cols_continuous] = df[cols_continuous].ffill()

    # Remueve nulos iniciales del primer paso temporal para asegurar valores estables
    initial_nans_count = df[cols_continuous].isnull().any(axis=1).sum()
    if initial_nans_count > 0:
        df = df.dropna(subset=cols_continuous)
        
    return df


def calculate_physical_features(df: pd.DataFrame) -> pd.DataFrame:
    """Calcula el viento vectorial, ciclos armónicos y depresión de rocío termodinámica."""
    # Codificación de ciclos meteorológicos diurnos y anuales
    df["hour_sin"] = np.sin(2 * np.pi * df.index.hour / 24.0)
    df["hour_cos"] = np.cos(2 * np.pi * df.index.hour / 24.0)
    df["month_sin"] = np.sin(2 * np.pi * df.index.month / 12.0)
    df["month_cos"] = np.cos(2 * np.pi * df.index.month / 12.0)

    # Conversión física del viento a componentes zonales y meridionales
    dd_rad = np.deg2rad(df["DD"])
    df["wind_u"] = -df["FF"] * np.sin(dd_rad)
    df["wind_v"] = -df["FF"] * np.cos(dd_rad)

    # Cálculo psicrométrico de depresión del punto de rocío de Magnus (evita HR = 0)
    hr_safe = df["HR"].clip(lower=0.1)
    alpha = ((17.625 * df["TT"]) / (243.04 + df["TT"])) + np.log(hr_safe / 100.0)
    dew_point = (243.04 * alpha) / (17.625 - alpha)
    df["dew_point_dep"] = df["TT"] - dew_point

    df = df.drop(columns=["DD", "FF"])
    return df


# ============================================================================
# EXTRACCIÓN DE LA CLIMATOLOGÍA Y ESTRATEGIA DE ANOMALÍAS
# ============================================================================

def apply_zero_leakage_climatology(df: pd.DataFrame, train_end_date: str) -> pd.DataFrame:
    """Extrae la climatología solo del set de entrenamiento para evitar fuga de datos futuros."""
    # Aislamiento temporal estricto de la partición de entrenamiento (2018-2023)
    df_train = df.loc[df.index <= pd.to_datetime(train_end_date)]

    # Cálculo de la señal climatológica determinista diaria y estacional (Mes, Hora)
    climatology = df_train.groupby([df_train.index.month, df_train.index.hour])["TT"].mean().rename("TT_climatology")

    # Mapeo y remoción de la señal sobre toda la serie temporal (2018-2025)
    df = df.join(climatology, on=[df.index.month, df.index.hour])
    df["TT_anomaly"] = df["TT"] - df["TT_climatology"]
    
    return df


# ============================================================================
# GENERACIÓN DE DIAGNÓSTICOS VISUALES CIENTÍFICOS
# ============================================================================

def generate_scientific_plots(df_clean: pd.DataFrame, df_tt_original: pd.Series, df_std: pd.DataFrame) -> None:
    """Genera las figuras de validación termodinámica y estadística."""
    # --- FIGURA 1: Validación de la imputación causal ---
    plt.figure(figsize=(6, 6))
    sns.kdeplot(df_tt_original.dropna(), label="Original", color="black", linewidth=1.5)
    sns.kdeplot(df_clean["TT"], label="Imputado (Causal)", color="crimson", linestyle="--", linewidth=1.5)
    plt.title("Distribución Estadística de la Temperatura del Aire", fontweight="bold")
    plt.xlabel("Temperatura del Aire (°C)")
    plt.ylabel("Densidad de Probabilidad")
    plt.legend(loc="upper right")
    plt.tight_layout()
    plt.savefig("fig_01_causal_imputation_distribution.png", dpi=300, bbox_inches="tight")
    plt.close()

    # --- FIGURA 2: Matriz de Correlación Spearman ---
    corr_features = ["TT", "HR", "PP", "RR", "wind_u", "wind_v", "dew_point_dep"]
    plt.figure(figsize=(6, 6))
    corr = df_clean[corr_features].corr(method="spearman")
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, annot=True, cmap="coolwarm", fmt=".2f", square=True, linewidths=.5, cbar_kws={"shrink": .8})
    plt.title("Matriz de Correlación de Spearman", fontweight="bold")
    plt.tight_layout()
    plt.savefig("fig_02_spearman_correlation_matrix.png", dpi=300, bbox_inches="tight")
    plt.close()

    # --- FIGURA 3: Auditoría de Variabilidad ---
    plt.figure(figsize=(6, 6))
    variables = df_std["Variable"].values
    std_values = df_std["Desviacion_Estandar"].values
    bars = plt.bar(variables, std_values, color="royalblue", edgecolor="black", alpha=0.8)
    plt.title("Auditoría de Variabilidad e Inconsistencias", fontweight="bold")
    plt.ylabel("Desviación Estándar")
    plt.xlabel("Variables Analizadas")
    plt.xticks(rotation=45)

    for bar, state in zip(bars, df_std["Estado"]):
        if state == "DESCARTE":
            bar.set_color("crimson")
            plt.text(bar.get_x() + bar.get_width() / 2.0, bar.get_height() + 0.1, "DESC", 
                     ha="center", va="bottom", color="crimson", fontsize=8, fontweight="bold")

    plt.tight_layout()
    plt.savefig("fig_03_variability_audit_justification.png", dpi=300, bbox_inches="tight")
    plt.close()

    # --- FIGURA 4: Serie de tiempo histórica ---
    plt.figure(figsize=(6, 6))
    plt.scatter(df_clean["datetime"], df_clean["TT"], s=1.5, alpha=0.15, color="darkorange")
    plt.title("Serie Temporal Histórica de Temperatura (Huayao)", fontweight="bold")
    plt.xlabel("Línea de Tiempo (2018 - 2025)")
    plt.ylabel("Temperatura Superficial (°C)")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig("fig_04_temperature_historical_scatter.png", dpi=300, bbox_inches="tight")
    plt.close()


# ============================================================================
# EJECUCIÓN DEL PIPELINE
# ============================================================================

def main() -> None:
    """Orquesta el pipeline físico-matemático de preprocesamiento de la EMA."""
    print("=" * 75)
    print("  PHASE 1: METEOROLOGICAL DATA ENGINEERING & FEATURE SELECTION AUDIT")
    print("=" * 75)

    df_raw = load_and_sanitize_data(INPUT_PATH)
    original_nulls, df_std = audit_variability_and_nulls(df_raw)
    
    df_temporal = build_uniform_timeline(df_raw, original_nulls)
    df_tt_original = df_temporal["TT"].copy()
    
    df_imputed = apply_causal_imputation(df_temporal)
    df_features = calculate_physical_features(df_imputed)
    df_climatological = apply_zero_leakage_climatology(df_features, TRAIN_END_DATE)

    # Reestructuración dimensional y aplicación del contrato de variables
    df_climatological = df_climatological.reset_index().rename(columns={"index": "datetime"})
    df_climatological["time_idx"] = np.arange(len(df_climatological))
    df_climatological["group_id"] = "Huayao_Station"

    clean_cols = [
        "datetime", "time_idx", "group_id", "TT", "HR", "PP", "RR", 
        "wind_u", "wind_v", "dew_point_dep", "TT_anomaly", "TT_climatology", 
        "hour_sin", "hour_cos", "month_sin", "month_cos"
    ]
    df_clean = df_climatological[clean_cols]
    
    # Generación de diagnósticos con las nuevas variables físicas
    generate_scientific_plots(df_clean, df_tt_original, df_std)
    df_clean.to_csv(OUTPUT_PATH, index=False)
    
    print(f"\n[INFO] Pipeline finalizado con éxito.")
    print(f"       Archivo exportado: {OUTPUT_PATH}")
    print(f"       Metadatos de nulos guardados en: {OUTPUT_METRIC_NULLS}")
    print(f"       Metadatos de desviación guardados en: {OUTPUT_METRIC_STD}")


if __name__ == "__main__":
    main()

  PHASE 1: METEOROLOGICAL DATA ENGINEERING & FEATURE SELECTION AUDIT

[INFO] Pipeline finalizado con éxito.
       Archivo exportado: dataset_huayao_preprocessed.csv
       Metadatos de nulos guardados en: metadata_auditoria_nulos.csv
       Metadatos de desviación guardados en: metadata_desviacion_estandar.csv
